In [ ]:
import json
import sqlite3
import time
import random
import threading
from datetime import datetime
import pandas as pd
import requests
from IPython.display import display, Markdown

from fastapi import FastAPI, HTTPException
from uvicorn import Config, Server

from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode
from opentelemetry.sdk.trace import TracerProvider, Resource
from opentelemetry.sdk.trace.export import SpanExporter, SimpleSpanProcessor
from opentelemetry.semconv.resource import ResourceAttributes
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor

# 1. BASE DE DATOS SQLITE PARA TELEMETRÍA
DB_FILE = "opentelemetry_live.db"

def init_db():
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute("DROP TABLE IF EXISTS telemetry_spans")
        conn.execute("""
            CREATE TABLE telemetry_spans (
                trace_id TEXT,
                span_id TEXT,
                parent_span_id TEXT,
                name TEXT,
                status_code TEXT,
                duration_ms REAL,
                attributes_json TEXT,
                events_json TEXT
            )
        """)
        conn.commit()

init_db()

# 2. EXPORTER DE OPENTELEMETRY DIRECTO A SQLITE
class SQLiteSpanExporter(SpanExporter):
    def export(self, spans) -> StatusCode:
        records = []
        for span in spans:
            if "http receive" in span.name or "http send" in span.name:
                continue

            duration_ms = (span.end_time - span.start_time) / 1_000_000.0
            events = [
                {
                    "name": ev.name,
                    "timestamp": datetime.fromtimestamp(ev.timestamp / 1e9).isoformat(),
                    "attributes": dict(ev.attributes or {})
                } for ev in span.events
            ]
            parent_id = f"{span.parent.span_id:016x}" if span.parent else None

            records.append((
                f"{span.context.trace_id:032x}",
                f"{span.context.span_id:016x}",
                parent_id,
                span.name,
                span.status.status_code.name,
                duration_ms,
                json.dumps(dict(span.attributes or {})),
                json.dumps(events)
            ))

        if records:
            with sqlite3.connect(DB_FILE) as conn:
                conn.executemany("""
                    INSERT INTO telemetry_spans VALUES (?,?,?,?,?,?,?,?)
                """, records)
                conn.commit()
        return StatusCode.OK

    def shutdown(self):
        pass

# 3. RESET COMPLETO DEL PROVEEDOR GLOBAL DE OPENTELEMETRY
# Resetea la instancia en memoria para evitar el error 'Overriding not allowed'
trace._TRACER_PROVIDER = None

resource = Resource.create(attributes={ResourceAttributes.SERVICE_NAME: "fastapi-ml-prod"})
tracer_provider = TracerProvider(resource=resource)
processor = SimpleSpanProcessor(SQLiteSpanExporter())
tracer_provider.add_span_processor(processor)
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("mlops.service")

# 4. APLICACIÓN FASTAPI E INSTRUMENTACIÓN
app = FastAPI()
FastAPIInstrumentor.uninstrument_app(app)  # Des-instrumentar si existía instancia previa
FastAPIInstrumentor.instrument_app(app)

def mock_feature_store_lookup(user_id: str):
    with tracer.start_as_current_span("feature_store_lookup") as span:
        span.set_attribute("user_id", user_id)
        time.sleep(random.uniform(0.01, 0.025))
        if user_id == "user_blocked":
            span.set_status(Status(StatusCode.ERROR, "Usuario bloqueado"))
            raise ValueError("Acceso denegado: Usuario en lista de fraude activa.")
        return {"avg_amount": 150.0}

def mock_model_inference(user_id: str, features: dict):
    with tracer.start_as_current_span("model_inference") as span:
        span.set_attribute("user_id", str(user_id))
        span.set_attribute("model_name", "xgboost_fraud_v2")
        time.sleep(random.uniform(0.05, 0.12))
        score = random.random()
        span.set_attribute("prediction_score", round(score, 4))
        return score

@app.post("/api/v1/predict")
async def predict_fraud(payload: dict):
    current_span = trace.get_current_span()
    user_id = payload.get("user_id")
    current_span.set_attribute("user_id", str(user_id))

    try:
        features = mock_feature_store_lookup(user_id)
        score = mock_model_inference(user_id, features)
        return {"user_id": user_id, "score": score}
    except ValueError as ve:
        current_span.record_exception(ve)
        current_span.set_status(Status(StatusCode.ERROR, str(ve)))
        raise HTTPException(status_code=403, detail=str(ve))

# 5. SERVIDOR EN PUERTO DINÁMICO
PORT = random.randint(8100, 8900)
server_config = Config(app=app, host="127.0.0.1", port=PORT, log_level="error")
server = Server(server_config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(1.5)

# 6. GENERACIÓN DE TRÁFICO
print(f"[PROD SIM] Enviando peticiones al servidor en puerto {PORT}...")
users_pool = ["user_101", "user_blocked", "user_204", "user_blocked", "user_305"]

for _ in range(10):
    uid = random.choice(users_pool)
    try:
        requests.post(f"http://127.0.0.1:{PORT}/api/v1/predict", json={"user_id": uid}, timeout=3)
    except Exception as e:
        print(f"Error en petición: {e}")

time.sleep(0.5)

# 7. EXTRACCIÓN Y VISUALIZACIÓN
with sqlite3.connect(DB_FILE) as conn:
    df = pd.read_sql_query("SELECT * FROM telemetry_spans", conn)

print("\n" + "="*90)
print("                 DATASET EXTRAÍDO DE TELEMETRÍA (NIVEL PRODUCCIÓN)")
print("="*90)
print(f"\nTotal Spans Registrados: {len(df)}\n")

if len(df) > 0:
    def parse_attributes(attr_str):
        try:
            data = json.loads(attr_str) if attr_str else {}
            return pd.Series({
                "extracted_user_id": data.get("user_id"),
                "model_name": data.get("model_name"),
                "prediction_score": data.get("prediction_score")
            })
        except Exception:
            return pd.Series({"extracted_user_id": None, "model_name": None, "prediction_score": None})

    parsed_attrs = df["attributes_json"].apply(parse_attributes)
    df_parsed = pd.concat([df, parsed_attrs], axis=1)

    display(Markdown("### 1. Métricas por Operación"))
    df_metrics = df_parsed.groupby("name").agg(
        peticiones=("trace_id", "count"),
        duracion_promedio_ms=("duration_ms", "mean"),
        duracion_max_ms=("duration_ms", "max"),
        duracion_min_ms=("duration_ms", "min")
    ).reset_index().rename(columns={"name": "operacion"})
    display(df_metrics)

    display(Markdown("### 2. Inferencias de ML Capturadas"))
    df_ml = df_parsed[df_parsed["name"] == "model_inference"][
        ["trace_id", "extracted_user_id", "model_name", "prediction_score", "duration_ms"]
    ].rename(columns={"extracted_user_id": "user_id"})
    display(df_ml)

    display(Markdown("### 3. Trazas de Error Capturadas"))
    df_errors = df_parsed[df_parsed["status_code"] == "ERROR"][
        ["trace_id", "name", "extracted_user_id", "status_code", "events_json"]
    ].rename(columns={"name": "operacion", "extracted_user_id": "user_id"})
    display(df_errors)